# SQL

이번 튜토리얼은 `create_sql_query_chain` 을 활용하여 생성한 체인으로 SQL 쿼리를 생성 후 실행, 그리고 답변 도출하는 방법에 대해서 다룹니다. 그리고, SQL Agent 와의 동작 방식의 차이도 알아보겠습니다.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase         # ① DB 연결 객체
from langchain_classic.chains import create_sql_query_chain   # ② SQL 문을 만드는 체인

# SQLite 파일 DB에 연결 - 연결 주소 형식
db = SQLDatabase.from_uri("sqlite:///data/finance.db")  

# 이 DB의 SQL 방언(종류) 이름 -> 함께 자주 찍는 것은 db.get_usable_table_names() 
print(db.dialect)

# 사용 가능한 테이블 이름 출력
print(db.get_usable_table_names())

sqlite
['accounts', 'customers', 'transactions']


LLM 객체를 생성하고 LLM 과 DB 를 매개변수로 입력하여 chain 을 생성합니다.

여기서 model 변경시 원활하게 동작하지 않을 수 있어, 이번 튜토리얼은 `gpt-3.5-turbo` 로 진행합니다.

In [4]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# LLM 과 DB 를 매개변수로 입력하여 chain 생성
chain = create_sql_query_chain(llm, db)

(옵션) 아래의 방식으로 Prompt 를 직접 지정할 수 있습니다.

직접 작성시 table_info 와 더불어 설명가능한 column description 을 추가할 수 있습니다.

In [5]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
     """Given an input question, first create a syntactically correct {dialect} query to run, then look at the results of the query and return the answer. Unless the user specifies in his question a specific number of examples he wishes to obtain, always limit your query to at most {top_k} results. You can order the results by a relevant column to return the most interesting examples in the database.
    Use the following format:

Question: "Question here"
SQLQuery: "SQL Query to run"
SQLResult: "Result of the SQLQuery"
Answer: "Final answer here"

Only use the following tables:
{table_info}

Here is the description of the columns in the tables:
`cust`: customer name
`prod`: product name
`trans`: transaction date

Question: {input}"""
).partial(dialect=db.dialect)

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

chain = create_sql_query_chain(llm, db, prompt)

chain 을 실행하면 DB 기반으로 쿼리를 생성합니다.

In [17]:
generated_sql_query = chain.invoke({"question": "고객의 이름을 나열하세요."})

print(generated_sql_query.__repr__())

"[('테디',), ('폴',), ('셜리',), ('민수',), ('지영',), ('은정',)]"


다음은 생성한 쿼리가 맞게 동작하는지 확인해 볼 차례입니다.

In [18]:
from langchain_community.tools.sql_database.tool import QuerySQLDatabaseTool

execute_query = QuerySQLDatabaseTool(db=db)

In [24]:
execute_query.invoke({"query": generated_sql_query})

'Error: (sqlite3.OperationalError) near "[(\'테디\',), (\'폴\',), (\'셜리\',), (\'민수\',), (\'지영\',), (\'은정\',)]": syntax error\n[SQL: [(\'테디\',), (\'폴\',), (\'셜리\',), (\'민수\',), (\'지영\',), (\'은정\',)]]\n(Background on this error at: https://sqlalche.me/e/21/e3q8)'

In [20]:
from langchain_community.tools.sql_database.tool import QuerySQLDatabaseTool

# 도구
execute_query = QuerySQLDatabaseTool(db=db)

# SQL 쿼리 생성 체인
write_query = create_sql_query_chain(llm, db)

# 생성한 쿼리를 실행하기 위한 체인을 생성합니다.
chain = write_query | execute_query

In [23]:
chain.invoke({"question": "테디의 이메일을 조회하세요"})

"[('teddy@example.com',)]"

## 답변을 LLM 으로 증강-생성

이전 단계에서 생성한 chain 을 사용하면 답변이 단답형 형식으로 출력됩니다. 이를 LCEL 문법의 체인으로 좀 더 자연스러운 답변을 받을 수 있도록 조정할 수 있습니다.

In [27]:
from operator import itemgetter
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

answer_prompt = PromptTemplate.from_template(
    """Given the following user question, corresponding SQL query, and SQL result, answer the user question.
Question: {question}
SQL Query: {query}
SQL Result: {result}
Answer: """
)

answer = answer_prompt | llm | StrOutputParser()

chain = (
    RunnablePassthrough.assign(query=write_query).assign(
        result=itemgetter("query") | execute_query
    )
    | answer
)

In [28]:
chain.invoke({"question": "테디의 transaction의 합계를 구하세요"})

'테디의 transaction의 합계는 -965.7 입니다.'

## Agent

Agent를 활용하여 Sql 쿼리를 생성하고 실행 결과를 답변으로 출력이 가능합니다.

In [31]:
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

db = SQLDatabase.from_uri("sqlite:///data/finance.db")

agent_executor = create_sql_agent(llm, db=db, agent_type="openai-tools", verbose=True)

In [32]:
agent_executor.invoke(
    {"input": "테디와 셜리의 transaction의 합계를 구하고 비교하세요"}
)



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


accounts, customers, transactions
Invoking: `sql_db_schema` with `{'table_names': 'transactions'}`



CREATE TABLE transactions (
	transaction_id INTEGER, 
	account_id INTEGER, 
	amount REAL, 
	transaction_date TEXT, 
	PRIMARY KEY (transaction_id), 
	FOREIGN KEY(account_id) REFERENCES accounts (account_id)
)

/*
3 rows from transactions table:
transaction_id	account_id	amount	transaction_date
1	1	74.79	2024-07-13
2	1	-224.1	2024-05-13
3	1	-128.9	2024-01-25
*/
Invoking: `sql_db_query` with `{'query': 'SELECT SUM(amount) AS total_amount FROM transactions WHERE account_id = 1'}`


[(-965.7,)]
Invoking: `sql_db_query` with `{'query': 'SELECT SUM(amount) AS total_amount FROM transactions WHERE account_id = 2'}`


[(743.13,)]테디의 거래 합계는 -965.7이고, 셜리의 거래 합계는 743.13입니다.

> Finished chain.


{'input': '테디와 셜리의 transaction의 합계를 구하고 비교하세요',
 'output': '테디의 거래 합계는 -965.7이고, 셜리의 거래 합계는 743.13입니다.'}